# tests/generate_fake_data — run the demo WITHOUT Synergy creds

Writes synthetic **teams** + **games** JSON (matching the Synergy `result` row shape) into the `raw_data`
Volume, exactly where `01_ingest` would land them. Then run `02_bronze_autoloader` → `03_silver` and the
whole medallion works offline — ideal for the handoff before live credentials are provisioned.

In [ ]:
# --- dual-mode setup: runs locally via Databricks Connect AND inside a Databricks Git Folder ---
import os, json
try:
    from dotenv import load_dotenv; load_dotenv()
except Exception:
    pass
try:
    spark  # already present inside a Databricks workspace notebook
except NameError:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.serverless().getOrCreate()

UC_CATALOG    = os.getenv("UC_CATALOG", "main")
UC_SCHEMA     = os.getenv("UC_SCHEMA", "synergy_baseball_demo")
BRONZE_SCHEMA = f"{UC_SCHEMA}_bronze"
SILVER_SCHEMA = f"{UC_SCHEMA}_silver"
GOLD_SCHEMA   = f"{UC_SCHEMA}_gold"
VOLUME_NAME   = "raw_data"
VOLUME_PATH   = f"/Volumes/{UC_CATALOG}/{BRONZE_SCHEMA}/{VOLUME_NAME}"
B, S, G = f"{UC_CATALOG}.{BRONZE_SCHEMA}", f"{UC_CATALOG}.{SILVER_SCHEMA}", f"{UC_CATALOG}.{GOLD_SCHEMA}"
print("target:", B, "|", S, "|", G)

In [ ]:
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()

# ensure the medallion schemas + raw Volume exist (idempotent — safe to re-run)
for sch in [BRONZE_SCHEMA, SILVER_SCHEMA, GOLD_SCHEMA]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {UC_CATALOG}.{sch}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {UC_CATALOG}.{BRONZE_SCHEMA}.{VOLUME_NAME}")

def upload_json(volume_path: str, payload) -> int:
    """Write a JSON payload to the Volume via the Files API (network upload, no FUSE needed)."""
    body = json.dumps(payload).encode("utf-8")
    w.files.upload(volume_path, body, overwrite=True)
    return len(body)
print("volume ready:", VOLUME_PATH)

In [ ]:
import random
random.seed(7)

LEAGUES = [("1","Triple-A","AAA"), ("2","Double-A","AA")]
fake_teams = []
for i in range(1, 21):
    lg = random.choice(LEAGUES)
    fake_teams.append({
        "id": f"T{i:04d}", "externalIdMlbam": 100000 + i, "name": f"Demo Club {i}",
        "nameAbbrev": f"DC{i}", "leagueId": lg[0],
        "league": {"name": lg[1], "nameAbbrev": lg[2]},
        "division": {"id": f"D{i%4}", "name": f"Division {i%4}"},
        "conference": {"id": f"C{i%2}", "name": f"Conference {i%2}"},
        "parentTeamId": None, "owner": {"id": "demo", "team": {"id": f"T{i:04d}"}},
        "datasourceId": "synthetic", "dateUpdated": "2024-04-01T00:00:00Z",
    })

import datetime as dt
fake_games = []
gid = 0
for d in range(7):
    day = (dt.date(2024, 4, 1) + dt.timedelta(days=d)).isoformat()
    for _ in range(8):
        gid += 1
        h, a = random.sample(fake_teams, 2)
        fake_games.append({
            "id": f"G{gid:05d}", "season": 2024, "date": f"{day}T18:00:00Z", "status": "Final",
            "homeTeam": {"id": h["id"], "name": h["name"], "division": h["division"]},
            "awayTeam": {"id": a["id"], "name": a["name"], "division": a["division"]},
            "owner": {"id": "demo", "team": {"id": h["id"]}},
            "datasourceId": "synthetic", "dateUpdated": f"{day}T23:00:00Z",
            "gameEventsDateUpdated": f"{day}T23:30:00Z",
        })

print(f"generated {len(fake_teams)} teams, {len(fake_games)} games")

In [ ]:
for entity, rows in [("teams", fake_teams), ("games", fake_games)]:
    path = f"{VOLUME_PATH}/{entity}/{entity}_synthetic.json"
    upload_json(path, rows)
    print(f"  wrote {len(rows)} {entity} -> {path}")
print("\nNow run 02_bronze_autoloader then 03_silver_transformations — no creds needed.")